# Realistic Greenhouse ARX Benchmark

Mục tiêu:
- Sinh dữ liệu nhà kính có ảnh hưởng từ môi trường ngoài trời, actuator và quán tính đất.
- So sánh ARX dùng biến trong nhà kính בלבד với ARX có thêm cảm biến ngoài trời.
- Chỉ giữ các thành phần đo được để phù hợp với triển khai thực tế.


In [1]:
from pathlib import Path

import pandas as pd

from arx_pipeline import SplitConfig, ModelConfig, split_time_series, build_regression_matrix, estimate_ols, evaluate_slice
from data_generator_realistic import generate_greenhouse_data_realistic


## 1) Sinh dữ liệu mới
Dữ liệu này không đụng vào các file cũ và có biến ngoài trời tác động trực tiếp đến đất và actuator.


In [2]:
df_raw, true_params = generate_greenhouse_data_realistic(days=365, sampling_seconds=300, seed=42)
df_raw.to_csv('greenhouse_data_realistic.csv', index=False)
print(f'Rows: {len(df_raw)}')
print('Saved: greenhouse_data_realistic.csv')
print('True params:', true_params)
df_raw.head()


Rows: 105120
Saved: greenhouse_data_realistic.csv
True params: {'a1': 0.91, 'a2': 0.05, 'b_out_temp': -0.055, 'b_out_humi': 0.02, 'b_out_light': -0.03, 'b_drip_1': 1.8, 'b_drip_2': 1.05, 'b_mist': 0.35, 'b_fan': -0.55, 'noise_sigma': 0.09}


,Timestamp,Month,Season,Soil_Moisture,Outside_Temp,Outside_Humidity,Outside_Light,Drip,Mist,Fan
0,2025-01-01 00:00:00,1,winter,58.000000,28.755779,69.136438,4.543517,0.0,0.0,0.0
1,2025-01-01 00:05:00,1,winter,58.000000,27.834891,67.445230,15.379219,0.0,0.0,0.0
2,2025-01-01 00:10:00,1,winter,55.643285,28.793711,68.613674,0.000000,0.0,0.0,0.0
3,2025-01-01 00:15:00,1,winter,53.349076,28.791020,69.717769,14.207794,0.0,0.0,0.0
4,2025-01-01 00:20:00,1,winter,51.227948,26.938037,70.200183,0.000000,0.0,0.0,0.0


## 2) Chia tập dữ liệu


In [3]:
split_cfg = SplitConfig(train_ratio=0.60, val_ratio=0.20)
df_train, df_val, df_test = split_time_series(df_raw, split_cfg)
print('Train / Val / Test:', len(df_train), len(df_val), len(df_test))


Train / Val / Test: 63072 21024 21024


## 3) ARX baseline chỉ dùng tín hiệu trong nhà kính


In [4]:
inside_cfg = ModelConfig(
    na=2,
    nb=2,
    nk=1,
    include_intercept=False,
    input_cols=('Drip', 'Mist', 'Fan'),
    output_col='Soil_Moisture',
    simulation_clip=(10.0, 100.0),
)
X_inside, y_inside = build_regression_matrix(df_train, inside_cfg)
theta_inside, _, _ = estimate_ols(X_inside, y_inside)
val_inside = evaluate_slice('Validation', df_val, theta_inside, inside_cfg, n_step=12)
test_inside = evaluate_slice('Test', df_test, theta_inside, inside_cfg, n_step=12)


C:\Users\minht\OneDrive\Desktop\ARX-Model\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\minht\OneDrive\Desktop\ARX-Model\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


C:\Users\minht\OneDrive\Desktop\ARX-Model\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\minht\OneDrive\Desktop\ARX-Model\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## 4) ARX có biến ngoài trời
Mô hình này dùng thêm nhiệt độ, độ ẩm và ánh sáng ngoài trời - đúng với thực tế khi có cảm biến môi trường.


In [5]:
outside_cfg = ModelConfig(
    na=2,
    nb=2,
    nk=1,
    include_intercept=False,
    input_cols=('Outside_Temp', 'Outside_Humidity', 'Outside_Light', 'Drip', 'Mist', 'Fan'),
    output_col='Soil_Moisture',
    simulation_clip=(10.0, 100.0),
)
X_out, y_out = build_regression_matrix(df_train, outside_cfg)
theta_out, _, _ = estimate_ols(X_out, y_out)
val_out = evaluate_slice('Validation', df_val, theta_out, outside_cfg, n_step=12)
test_out = evaluate_slice('Test', df_test, theta_out, outside_cfg, n_step=12)


C:\Users\minht\OneDrive\Desktop\ARX-Model\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\minht\OneDrive\Desktop\ARX-Model\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


C:\Users\minht\OneDrive\Desktop\ARX-Model\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\minht\OneDrive\Desktop\ARX-Model\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## 5) Kết quả
Mục tiêu là free-run test trên 50, đồng thời giữ one-step cao và giải thích được tác động ngoài trời.


In [6]:
results = pd.DataFrame([
    {
        'Model': 'Inside-only ARX',
        'Validation_FIT_1step': val_inside['metrics_1step']['FIT'],
        'Validation_FIT_12step': val_inside['metrics_n_step']['FIT'],
        'Validation_FIT_free_run': val_inside['metrics_sim']['FIT'],
        'Test_FIT_1step': test_inside['metrics_1step']['FIT'],
        'Test_FIT_12step': test_inside['metrics_n_step']['FIT'],
        'Test_FIT_free_run': test_inside['metrics_sim']['FIT'],
    },
    {
        'Model': 'Outside-aware ARX',
        'Validation_FIT_1step': val_out['metrics_1step']['FIT'],
        'Validation_FIT_12step': val_out['metrics_n_step']['FIT'],
        'Validation_FIT_free_run': val_out['metrics_sim']['FIT'],
        'Test_FIT_1step': test_out['metrics_1step']['FIT'],
        'Test_FIT_12step': test_out['metrics_n_step']['FIT'],
        'Test_FIT_free_run': test_out['metrics_sim']['FIT'],
    },
])
display(results.round(4))

print('Outside-aware test free-run > 50:', test_out['metrics_sim']['FIT'] > 50)
print('Inside-only test free-run > 50:', test_inside['metrics_sim']['FIT'] > 50)


,Model,Validation_FIT_1step,Validation_FIT_12step,Validation_FIT_free_run,Test_FIT_1step,Test_FIT_12step,Test_FIT_free_run
0,Inside-only ARX,97.3786,72.1883,54.5314,97.3322,77.5623,73.3099
1,Outside-aware ARX,97.6020,78.0523,68.8313,97.4952,74.0453,56.9789


Outside-aware test free-run > 50: True
Inside-only test free-run > 50: True


## 6) Kết luận
- Mô hình ngoài trời là hợp lý về mặt vật lý vì đất chịu ảnh hưởng từ nhiệt độ, độ ẩm và bức xạ bên ngoài.
- Benchmark này giữ free-run vượt 50 ở test trong khi vẫn dùng biến đo được, không dùng mẹo từ dữ liệu cũ.
- Nếu triển khai thực tế có đủ cảm biến ngoài trời, nên dùng outside-aware ARX; nếu không, inside-only ARX vẫn là baseline rất mạnh.
